# Day 19 — Register Django API URLs and Test Player Endpoints

**Project:** IPL Player Performance Analytics and Prediction  
**Goal:** Connect `ipl_api.urls` to the Django project URL configuration, run Django checks, start the development server, and test the Player List and Player Detail endpoints.

> Run this notebook from the repository root in the `ipl-player-performance` environment. It inspects files before editing and backs up `config/urls.py` before making a change. It does not change database tables or model artifacts.

## Expected structure

```text
ipl-player-performance/
└── django_api/
    ├── manage.py
    ├── config/
    │   ├── settings.py
    │   └── urls.py
    └── ipl_api/
        ├── views.py
        └── urls.py
```

If your actual structure differs, the discovery cell will show the detected paths.

In [1]:
from pathlib import Path
import os
import sys
import subprocess
import time
import json
import csv
from datetime import datetime

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "django_api").is_dir():
    candidates = [p for p in [REPO_ROOT, *REPO_ROOT.parents] if (p / "django_api").is_dir()]
    if candidates:
        REPO_ROOT = candidates[0]

DJANGO_ROOT = REPO_ROOT / "django_api"
MANAGE_PY = DJANGO_ROOT / "manage.py"
CONFIG_URLS = DJANGO_ROOT / "config" / "urls.py"
APP_URLS = DJANGO_ROOT / "ipl_api" / "urls.py"
APP_VIEWS = DJANGO_ROOT / "ipl_api" / "views.py"
DOCS_DIR = REPO_ROOT / "docs"
REPORTS_DIR = REPO_ROOT / "reports" / "day19"
DOCS_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print("Repository root:", REPO_ROOT)
print("Django root:", DJANGO_ROOT)
for p in [MANAGE_PY, CONFIG_URLS, APP_URLS, APP_VIEWS, DJANGO_ROOT / "config" / "settings.py"]:
    print(("FOUND  " if p.exists() else "MISSING"), p)


Repository root: D:\Git\Personal\ipl-player-performance
Django root: D:\Git\Personal\ipl-player-performance\django_api
FOUND   D:\Git\Personal\ipl-player-performance\django_api\manage.py
FOUND   D:\Git\Personal\ipl-player-performance\django_api\config\urls.py
FOUND   D:\Git\Personal\ipl-player-performance\django_api\ipl_api\urls.py
FOUND   D:\Git\Personal\ipl-player-performance\django_api\ipl_api\views.py
FOUND   D:\Git\Personal\ipl-player-performance\django_api\config\settings.py


In [2]:
# Preflight: stop if required Django files are missing.
required = [MANAGE_PY, CONFIG_URLS, APP_URLS, APP_VIEWS, DJANGO_ROOT / "config" / "settings.py"]
missing = [str(p) for p in required if not p.exists()]
if missing:
    raise FileNotFoundError(
        "Required project files are missing. Confirm you are using the correct repository and that Day 18 completed:\n"
        + "\n".join(missing)
    )

settings_text = (DJANGO_ROOT / "config" / "settings.py").read_text(encoding="utf-8")
project_urls_text = CONFIG_URLS.read_text(encoding="utf-8")
app_urls_text = APP_URLS.read_text(encoding="utf-8")

print("\n--- django_api/ipl_api/urls.py ---")
print(app_urls_text)
print("\n--- django_api/config/urls.py ---")
print(project_urls_text)
print("\nipl_api in INSTALLED_APPS:", "ipl_api" in settings_text)



--- django_api/ipl_api/urls.py ---
from django.urls import path
from .views import PlayerListAPIView, PlayerDetailAPIView

urlpatterns = [
    path("players/", PlayerListAPIView.as_view(), name="player-list"),
    path("players/<str:player_name>/", PlayerDetailAPIView.as_view(), name="player-detail"),
]


--- django_api/config/urls.py ---
"""
URL configuration for config project.

The `urlpatterns` list routes URLs to views. For more information please see:
    https://docs.djangoproject.com/en/6.1/topics/http/urls/
Examples:
Function views
    1. Add an import:  from my_app import views
    2. Add a URL to urlpatterns:  path('', views.home, name='home')
Class-based views
    1. Add an import:  from other_app.views import Home
    2. Add a URL to urlpatterns:  path('', Home.as_view(), name='home')
Including another URLconf
    1. Import the include() function: from django.urls import include, path
    2. Add a URL to urlpatterns:  path('blog/', include('blog.urls'))
"""

from django.c

In [3]:
# Register app URLs idempotently: repair the include import even if the route already exists.
import re

updated = project_urls_text
route_pattern = r"path\s*\(\s*['\"]api/['\"]\s*,\s*include\s*\(\s*['\"]ipl_api\.urls['\"]\s*\)\s*\)"
has_route = re.search(route_pattern, updated) is not None

# Ensure include is imported from django.urls.
import_match = re.search(r"from\s+django\.urls\s+import\s+([^\n]+)", updated)
if import_match:
    imported_names = [name.strip() for name in import_match.group(1).split(",")]
    if "include" not in imported_names:
        new_imports = ", ".join(imported_names + ["include"])
        updated = updated[:import_match.start(1)] + new_imports + updated[import_match.end(1):]
else:
    updated = "from django.urls import include, path\n" + updated

if not has_route:
    pattern = re.search(r"urlpatterns\s*=\s*\[", updated)
    if not pattern:
        raise ValueError("Could not locate urlpatterns = [ in config/urls.py. No URL route was inserted.")
    insertion = "\n    path('api/', include('ipl_api.urls')),  # Day 19: IPL API routes"
    updated = updated[:pattern.end()] + insertion + updated[pattern.end():]

if updated != project_urls_text:
    backup = CONFIG_URLS.with_name(f"urls.py.day19_backup_{datetime.now().strftime('%Y%m%d_%H%M%S')}")
    backup.write_text(project_urls_text, encoding="utf-8")
    CONFIG_URLS.write_text(updated, encoding="utf-8")
    print("Updated project URL configuration. Backup:", backup)
else:
    print("Project URL configuration already contains the import and route; no changes needed.")

project_urls_text = CONFIG_URLS.read_text(encoding="utf-8")
print("\nCurrent project URL configuration:")
print(project_urls_text)


Project URL configuration already contains the import and route; no changes needed.

Current project URL configuration:
"""
URL configuration for config project.

The `urlpatterns` list routes URLs to views. For more information please see:
    https://docs.djangoproject.com/en/6.1/topics/http/urls/
Examples:
Function views
    1. Add an import:  from my_app import views
    2. Add a URL to urlpatterns:  path('', views.home, name='home')
Class-based views
    1. Add an import:  from other_app.views import Home
    2. Add a URL to urlpatterns:  path('', Home.as_view(), name='home')
Including another URLconf
    1. Import the include() function: from django.urls import include, path
    2. Add a URL to urlpatterns:  path('blog/', include('blog.urls'))
"""

from django.contrib import admin
from django.urls import path, include

urlpatterns = [
    path('api/', include('ipl_api.urls')),  # Day 19: IPL API routes
    path("admin/", admin.site.urls),
]



In [4]:
# Run Django's system check from the directory containing manage.py.
check = subprocess.run(
    [sys.executable, str(MANAGE_PY), "check"],
    cwd=str(DJANGO_ROOT),
    capture_output=True,
    text=True,
)
print("Command: python manage.py check")
print("Return code:", check.returncode)
print(check.stdout)
if check.stderr:
    print(check.stderr)
if check.returncode != 0:
    raise RuntimeError("Django system check failed. Resolve the reported issue before endpoint testing.")


Command: python manage.py check
Return code: 0
System check identified no issues (0 silenced).



In [5]:
# Inspect URL patterns with Django's show_urls alternative: use URL resolver directly.
# This does not start a server or make HTTP requests.
env = os.environ.copy()
env["DJANGO_SETTINGS_MODULE"] = "config.settings"
snippet = r'''
import django
django.setup()
from django.urls import get_resolver
def walk(patterns, prefix=""):
    for p in patterns:
        route = prefix + str(p.pattern)
        if hasattr(p, "url_patterns"):
            walk(p.url_patterns, route)
        else:
            print(route, "->", getattr(p.callback, "__name__", str(p.callback)))
walk(get_resolver().url_patterns)
'''
url_report = subprocess.run(
    [sys.executable, "-c", snippet],
    cwd=str(DJANGO_ROOT),
    env=env,
    capture_output=True,
    text=True,
)
print("URL resolver return code:", url_report.returncode)
print(url_report.stdout)
if url_report.stderr:
    print(url_report.stderr)
if url_report.returncode != 0:
    raise RuntimeError("Could not inspect URL patterns.")


URL resolver return code: 0
api/players/ -> view
api/players/<str:player_name>/ -> view
admin/ -> index
admin/login/ -> login
admin/logout/ -> logout
admin/password_change/ -> password_change
admin/password_change/done/ -> password_change_done
admin/autocomplete/ -> autocomplete_view
admin/jsi18n/ -> i18n_javascript
admin/r/<path:content_type_id>/<path:object_id>/ -> shortcut
admin/auth/group/ -> changelist_view
admin/auth/group/add/ -> add_view
admin/auth/group/<path:object_id>/history/ -> history_view
admin/auth/group/<path:object_id>/delete/ -> delete_view
admin/auth/group/<path:object_id>/change/ -> change_view
admin/auth/group/<path:object_id>/ -> view
admin/auth/user/<id>/password/ -> user_change_password
admin/auth/user/ -> changelist_view
admin/auth/user/add/ -> add_view
admin/auth/user/<path:object_id>/history/ -> history_view
admin/auth/user/<path:object_id>/delete/ -> delete_view
admin/auth/user/<path:object_id>/change/ -> change_view
admin/auth/user/<path:object_id>/ -> vie

## Start the server and test endpoints

Open a **separate PowerShell terminal** so the server remains running. From the repository root:

```powershell
conda activate ipl-player-performance
cd D:\Git\Personal\ipl-player-performance\django_api
python manage.py runserver
```

Keep that terminal open. Return to this notebook and run the next cell. It sends HTTP GET requests to the local development server. If your server uses another port, update `BASE_URL`.

In [6]:
# HTTP smoke tests. Run this cell only after `runserver` is running in another terminal.
import urllib.request
import urllib.error

BASE_URL = "http://127.0.0.1:8000"
# Expected from the Day 18 player API design. Change only if your generated ipl_api/urls.py uses other paths.
ENDPOINTS = [
    ("Player list", "/api/players/"),
]

http_results = []
for label, path in ENDPOINTS:
    url = BASE_URL + path
    try:
        req = urllib.request.Request(url, headers={"Accept": "application/json"})
        with urllib.request.urlopen(req, timeout=10) as response:
            body = response.read().decode("utf-8", errors="replace")
            result = {
                "label": label, "url": url, "status_code": response.status,
                "content_type": response.headers.get("Content-Type", ""),
                "response_preview": body[:500], "error": ""
            }
    except urllib.error.HTTPError as e:
        body = e.read().decode("utf-8", errors="replace")
        result = {
            "label": label, "url": url, "status_code": e.code,
            "content_type": e.headers.get("Content-Type", ""),
            "response_preview": body[:500], "error": f"HTTPError: {e}"
        }
    except Exception as e:
        result = {
            "label": label, "url": url, "status_code": None,
            "content_type": "", "response_preview": "", "error": repr(e)
        }
    http_results.append(result)
    print(f"\n{label}: {url}")
    print("Status:", result["status_code"], "| Content-Type:", result["content_type"])
    print("Error:", result["error"])
    print("Preview:", result["response_preview"])




Player list: http://127.0.0.1:8000/api/players/
Status: None | Content-Type: 
Error: URLError(ConnectionRefusedError(10061, 'No connection could be made because the target machine actively refused it', None, 10061, None))
Preview: 


In [7]:
# Add the player-detail endpoint test using a real name from the complete player-list response.
import urllib.parse

def extract_names(payload):
    if isinstance(payload, dict):
        for key in ("results", "players", "data"):
            if isinstance(payload.get(key), list):
                payload = payload[key]
                break
        else:
            payload = [payload]
    if not isinstance(payload, list):
        return []
    names = []
    for row in payload:
        if isinstance(row, dict):
            for key in ("player_name", "name", "batter", "bowler", "full_name"):
                if row.get(key):
                    names.append(str(row[key]))
                    break
    return names

list_result = next((r for r in http_results if r["label"] == "Player list"), None)
detail_result = None
if list_result and list_result["status_code"] == 200:
    try:
        req = urllib.request.Request(list_result["url"], headers={"Accept": "application/json"})
        with urllib.request.urlopen(req, timeout=10) as response:
            full_payload = json.loads(response.read().decode("utf-8", errors="replace"))
        names = extract_names(full_payload)
    except Exception as e:
        names = []
        print("Could not read complete Player List JSON:", repr(e))
    if names:
        player_name = names[0]
        detail_path = "/api/players/" + urllib.parse.quote(player_name, safe="") + "/"
        url = BASE_URL + detail_path
        try:
            req = urllib.request.Request(url, headers={"Accept": "application/json"})
            with urllib.request.urlopen(req, timeout=10) as response:
                body = response.read().decode("utf-8", errors="replace")
                detail_result = {
                    "label": "Player detail", "url": url, "status_code": response.status,
                    "content_type": response.headers.get("Content-Type", ""),
                    "response_preview": body[:500], "error": ""
                }
        except Exception as e:
            detail_result = {"label": "Player detail", "url": url, "status_code": getattr(e, "code", None),
                             "content_type": "", "response_preview": "", "error": repr(e)}
        http_results.append(detail_result)
        print("Player detail URL:", url)
        print("Status:", detail_result["status_code"])
        print("Error:", detail_result["error"])
        print("Preview:", detail_result["response_preview"])
    else:
        print("Could not extract a player name. Inspect the Player List response and test a name manually.")
else:
    print("Skipping detail test until Player List returns HTTP 200.")


Skipping detail test until Player List returns HTTP 200.


In [8]:
# Save the actual test results and generate the Day 19 documentation automatically.
results_csv = REPORTS_DIR / "endpoint_smoke_test.csv"
with results_csv.open("w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["label", "url", "status_code", "content_type", "response_preview", "error"])
    writer.writeheader()
    writer.writerows(http_results)

status_rows = [
    ("project_root", str(REPO_ROOT)),
    ("django_root", str(DJANGO_ROOT)),
    ("manage_py_check_return_code", str(check.returncode)),
    ("project_url_config", str(CONFIG_URLS)),
    ("app_url_config", str(APP_URLS)),
    ("endpoint_test_count", str(len(http_results))),
    ("endpoint_test_results", "; ".join(f"{r['label']}={r['status_code']}" for r in http_results)),
    ("generated_at", datetime.now().isoformat(timespec="seconds")),
]
with (REPORTS_DIR / "task_status.csv").open("w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["item", "value"])
    writer.writerows(status_rows)

lines = [
    "# Day 19 — Django API URL Registration and Endpoint Testing",
    "",
    f"- Generated: {datetime.now().isoformat(timespec='seconds')}",
    f"- Repository root: `{REPO_ROOT}`",
    f"- Django root: `{DJANGO_ROOT}`",
    f"- `manage.py check` return code: `{check.returncode}`",
    f"- Project URL config: `{CONFIG_URLS}`",
    f"- App URL config: `{APP_URLS}`",
    "",
    "## HTTP smoke-test results",
    "",
    "| Endpoint | HTTP status | Content type | Error |",
    "|---|---:|---|---|",
]
for r in http_results:
    lines.append(f"| {r['label']} ({r['url']}) | {r['status_code']} | {r['content_type']} | {r['error'].replace('|', '/')} |")
lines += [
    "",
    "## Output files",
    "",
    f"- `{results_csv}`",
    f"- `{REPORTS_DIR / 'task_status.csv'}`",
    "",
    "## Interpretation",
    "",
    "- HTTP 200 means the endpoint responded successfully; inspect the JSON payload for correctness.",
    "- HTTP 404 usually means the URL route or path does not match the generated app URL patterns.",
    "- HTTP 500 means the request reached Django but server-side code failed; inspect the `runserver` terminal traceback.",
    "- Connection refused means the development server is not running at `BASE_URL`.",
    "",
    "## Manual confirmation still required",
    "",
    "- Confirm `config/urls.py` includes `path('api/', include('ipl_api.urls'))`.",
    "- Confirm endpoint JSON contains expected player fields and valid aggregates.",
    "- Do not treat HTTP 200 alone as proof of analytical correctness."
]
(DOCS_DIR / "day19_api_url_registration_and_testing.md").write_text("\n".join(lines) + "\n", encoding="utf-8")
print("Saved:", results_csv)
print("Saved:", REPORTS_DIR / "task_status.csv")
print("Saved:", DOCS_DIR / "day19_api_url_registration_and_testing.md")
print("\nDay 19 documentation was generated from the outputs observed in this notebook.")


Saved: D:\Git\Personal\ipl-player-performance\reports\day19\endpoint_smoke_test.csv
Saved: D:\Git\Personal\ipl-player-performance\reports\day19\task_status.csv
Saved: D:\Git\Personal\ipl-player-performance\docs\day19_api_url_registration_and_testing.md

Day 19 documentation was generated from the outputs observed in this notebook.


## Day 19 completion checklist

- [ ] All preflight files were found.
- [ ] `python manage.py check` returned code 0.
- [ ] URL resolver showed the expected `/api/players/` route.
- [ ] Player List returned HTTP 200 and valid JSON.
- [ ] Player Detail returned HTTP 200 for a real player name.
- [ ] Review `docs/day19_api_url_registration_and_testing.md` and `reports/day19/endpoint_smoke_test.csv`.

**Important:** Do not mark the API tests complete if the server returns an error. Save the traceback and endpoint response so the issue can be fixed before Day 20.